## Task 1
Загрузите данные из конкурса Don'tGetKicked . Разработайте схему разделения на обучающую, валидационную и тестовую выборки.
Используйте поле "PurchDate" для разделения выборки: тестовая дата должна быть позже даты валидации, то же самое относится к валидации и обучению: train.PurchDate < valid.PurchDate < test.PurchDate.
Используйте первые 33% данных для обучения, последние 33% — для тестирования, а средние 33% — для валидационного набора данных. Тестовый набор данных используйте только в конце!
Используйте LabelEncoder или OneHotEncoder из библиотеки sklearn для предварительной обработки категориальных переменных. Будьте осторожны с утечкой данных (обучите Encoder на обучающей выборке и примените его к валидационной и тестовой выборкам). Рассмотрите другой подход к кодированию, если вы обнаружите новые категориальные значения в валидационной и тестовой выборках (не встречавшиеся в обучающей выборке), например: https://contrib.scikit-learn.org/category_encoders/count.html

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier as SklearnDecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("data/training.csv")
df["PurchDate"] = pd.to_datetime(df["PurchDate"]).dt.date
df = df.sort_values("PurchDate").reset_index(drop=True)
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389,0,2009-01-05,MANHEIM,2007,2,CHRYSLER,PACIFICA FWD 3.8L V6,Bas,4D SPORT,...,9906.0,11657.0,NaN,NaN,3453,80022,CO,6770.0,0,1389
1,32406,0,2009-01-05,MANHEIM,2005,4,FORD,FREESTAR FWD V6 3.9L,SES,4D PASSENGER 3.9L SES,...,5801.0,6949.0,NaN,NaN,22916,80022,CO,6160.0,0,941
2,32407,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SE,4D SEDAN SE,...,4169.0,5114.0,NaN,NaN,3453,80022,CO,4250.0,0,1155
3,32408,0,2009-01-05,MANHEIM,2006,3,CHEVROLET,TRAILBLAZER EXT 4WD,LS,4D SUV 4.2L,...,10438.0,12158.0,NaN,NaN,22916,80022,CO,8180.0,0,1703
4,32409,0,2009-01-05,MANHEIM,2004,5,FORD,TAURUS 3.0L V6 EFI,SES,4D SEDAN SES DURATEC,...,4139.0,5351.0,NaN,NaN,22916,80022,CO,4900.0,0,825


In [3]:
unique_dates =sorted(df["PurchDate"].unique())
n = len(unique_dates)
d1 = unique_dates[n // 3]
d2 = unique_dates[2 * n // 3]
train_df = df[df["PurchDate"] < d1]
valid_df = df[(df["PurchDate"] >= d1) & (df["PurchDate"] < d2)]
test_df  = df[df["PurchDate"] >= d2]

In [4]:
print(train_df["PurchDate"].max())
print(valid_df["PurchDate"].min())

print(valid_df["PurchDate"].max())
print(test_df["PurchDate"].min())

2009-09-01
2009-09-02
2010-04-30
2010-05-03


In [5]:
target = "IsBadBuy"

X_train = train_df.drop(columns=[target, "PurchDate"])
X_valid = valid_df.drop(columns=[target, "PurchDate"])
X_test  = test_df.drop(columns=[target, "PurchDate"])

y_train = train_df[target]
y_valid = valid_df[target]
y_test  = test_df[target]

In [6]:
cat_cols = X_train.select_dtypes(include=["object", "string"]).columns
cat_cols

Index(['Auction', 'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
       'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'PRIMEUNIT',
       'AUCGUART', 'VNST'],
      dtype='str')

In [7]:

encoders = {}

for col in cat_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))

    encoders[col] = le

    X_valid[col] = X_valid[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

    X_test[col] = X_test[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

In [8]:
imputer = SimpleImputer(strategy="most_frequent")

datasets = {
    "train": X_train,
    "valid": X_valid,
    "test": X_test
}

cols = X_train.columns
result = {}

for name, X in datasets.items():
    if name == "train":
        arr = imputer.fit_transform(X)   
    else:
        arr = imputer.transform(X)

    result[name] = pd.DataFrame(
        arr,
        columns=cols,
        index=X.index
    )

X_train = result["train"]
X_valid = result["valid"]
X_test  = result["test"]

In [9]:
X_train.head()

,RefId,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,Color,Transmission,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,32389.0,1.0,2007.0,2.0,4.0,521.0,7.0,238.0,2.0,0.0,...,9906.0,11657.0,1.0,1.0,3453.0,80022.0,3.0,6770.0,0.0,1389.0
1,32406.0,1.0,2005.0,4.0,6.0,292.0,76.0,114.0,13.0,0.0,...,5801.0,6949.0,1.0,1.0,22916.0,80022.0,3.0,6160.0,0.0,941.0
2,32407.0,1.0,2004.0,5.0,5.0,659.0,74.0,196.0,13.0,0.0,...,4169.0,5114.0,1.0,1.0,3453.0,80022.0,3.0,4250.0,0.0,1155.0
3,32408.0,1.0,2006.0,3.0,3.0,715.0,49.0,295.0,14.0,0.0,...,10438.0,12158.0,1.0,1.0,22916.0,80022.0,3.0,8180.0,0.0,1703.0
4,32409.0,1.0,2004.0,5.0,6.0,682.0,76.0,210.0,4.0,0.0,...,4139.0,5351.0,1.0,1.0,22916.0,80022.0,3.0,4900.0,0.0,825.0


## Task 2
Создайте класс Python для классификатора дерева решений и регрессора дерева решений (функция потерь MSE).

Он должен поддерживать методы fit , predict_proba и predict . Кроме того, максимальная глубина (max_depth) должна быть параметром вашего класса. Используйте критерий Джини для выбора разбиения.

Вот план действий:
~~~
model = DecisionTreeClassifier(max_depth=7)
model.fit(Xtrain, ytrain)
model.predict_proba(Xvalid)
~~~



- Создайте отдельный класс для узла. Он должен уметь хранить данные (примеры признаков и целевые значения), вычислять коэффициент Джини и иметь указатели на дочерние узлы (левый и правый узлы). Для регрессора используйте стандартное отклонение вместо коэффициента Джини.
- Реализуйте функцию, которая находит наилучшее возможное разделение в текущем узле.
- Объедините предыдущие шаги в работающий классификатор на основе дерева решений.
- Реализуйте дополнительное рандомизированное дерево, разработав еще одну функцию для поиска наилучшего разделения.

In [10]:
class Node():
    def __init__(self,
                feature_idx=None,
                threshold=None,
                left=None,
                right=None,
                prediction=None,
                proba=None,
                depth=0):
        
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.prediction = prediction
        self.depth = depth
        self.proba = proba
        
    
        
        
class DecisionTree():

    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None


    def _find_best_split(self, X, y):
        pass
        
    def _predict_res(self, y):
        pass
    
    def _build_tree(self, X, y):
        self.root = Node()
        stack = [(self.root, list(range(len(y))))]

        while stack:
            node, idx = stack.pop()
            if node.depth < self.max_depth and len(idx) > self.min_samples_split:
                best_feature, best_threshold = self._find_best_split(X[idx], y[idx])   
                if best_threshold is not None:
                    node.feature_idx = best_feature
                    
                    node.left = Node(depth=node.depth + 1)
                    node.right = Node(depth=node.depth + 1)
                    
                    node.threshold = best_threshold     
                    left_idx = []
                    right_idx = []
                    for new_idx in idx:
                        if X[new_idx, node.feature_idx] <= node.threshold:
                            left_idx.append(new_idx)
                        else:
                            right_idx.append(new_idx)
                    if len(left_idx) == 0 or len(right_idx) == 0:
                        node.prediction, node.proba = self._predict_res(y[idx])
                        continue
                    stack.append((node.left, left_idx))
                    stack.append((node.right, right_idx))
                else: 
                    node.prediction, node.proba = self._predict_res(y[idx])
            else:
                node.prediction, node.proba = self._predict_res(y[idx])
            
    
    def fit(self, X_data, y_data):
        X_data = np.asarray(X_data)
        y_data = np.asarray(y_data)
        self._build_tree(X_data, y_data)
        return self

    def predict(self, X):
        X = np.asarray(X)
        predictions = []
        for x in X:
            node = self.root
    
            while node.left is not None and node.right is not None:
                if x[node.feature_idx] <= node.threshold:
                    node = node.left
                else:
                    node = node.right
            predictions.append(node.prediction)
        return np.array(predictions)
    
    def predict_proba(self, X):
        pass

класс для деревьяв с разделением при помощи джини или медианы

In [11]:
class DecisionTreeClassifier(DecisionTree):
    def __init__(self, max_depth=5, min_samples_split=2, method_split="gini"):
        super().__init__(max_depth=max_depth, min_samples_split=min_samples_split)
        self.n_classes_ = None
        self.method_split = method_split
        self.classes_ = None

    def _gini(self, y):
        if len(y) == 0:
            return 0.0
        counts = np.bincount(y, minlength=self.n_classes_)
        probs = counts / len(y)
        return 1.0 - np.sum(probs ** 2)

    def _find_best_split(self, X, y):
        if self.method_split == "gini":
            return self._find_best_split_gini(X, y)
        elif self.method_split == "median":
            return self._find_best_split_median(X, y)
        else:
            raise ValueError("method_split must be 'gini' or 'median'")

    def _find_best_split_gini(self, X, y):
        n_objects, n_features = X.shape
    
        best_feature = None
        best_threshold = None
        best_gain = -1.0
    
        parent_gini = self._gini(y)
        
        classes = np.unique(y)
    
        if len(classes) < 2:
            return None, None
    
        for feature_idx in range(n_features):
    
            values = np.unique(X[:, feature_idx])
    
            if len(values) < 2:
                continue
    
            thresholds = (values[:-1] + values[1:]) / 2.0
    
            for threshold in thresholds:
    
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
    
                y_left = y[left_mask]
                y_right = y[right_mask]
    
                if len(y_left) == 0 or len(y_right) == 0:
                    continue
    
                gini_left = self._gini(y_left)
                gini_right = self._gini(y_right)
    
                weighted = (
                    len(y_left)/len(y) * gini_left +
                    len(y_right)/len(y) * gini_right
                )
    
                gain = parent_gini - weighted
    
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
    
        return best_feature, best_threshold


    def _find_best_split_median(self, X, y):
        n_objects, n_features = X.shape
    
        best_feature = None
        best_threshold = None
        best_gain = -1.0
    
        parent_gini = self._gini(y)
        classes = np.unique(y)
    
        if len(classes) < 2:
            return None, None
    
        for feature_idx in range(n_features):
            x = X[:, feature_idx]
    
            for i in range(len(classes)):
                for j in range(i + 1, len(classes)):
                    c1 = classes[i]
                    c2 = classes[j]
    
                    x1 = x[y == c1]
                    x2 = x[y == c2]
    
                    if len(x1) == 0 or len(x2) == 0:
                        continue
    
                    m1 = np.median(x1)
                    m2 = np.median(x2)
                    threshold = (m1 + m2) / 2.0
    
                    left_mask = x <= threshold
                    right_mask = x > threshold
    
                    y_left = y[left_mask]
                    y_right = y[right_mask]
    
                    if len(y_left) == 0 or len(y_right) == 0:
                        continue
    
                    gini_left = self._gini(y_left)
                    gini_right = self._gini(y_right)
    
                    weighted = (
                        len(y_left) / len(y) * gini_left +
                        len(y_right) / len(y) * gini_right
                    )
    
                    gain = parent_gini - weighted
    
                    if gain > best_gain:
                        best_gain = gain
                        best_feature = feature_idx
                        best_threshold = threshold
    
        return best_feature, best_threshold
 

    def _predict_res(self, y):
        counts = np.bincount(y, minlength=self.n_classes_)
        proba = counts / counts.sum()
        prediction_idx = np.argmax(proba)
        prediction = self.classes_[prediction_idx]
        return prediction, proba

    def fit(self, X_data, y_data):
        X_data = np.asarray(X_data)
        y_data = np.asarray(y_data, dtype=int)
        self.classes_, y_encoded = np.unique(y_data, return_inverse=True)
        self.n_classes_ = len(self.classes_)
        self._build_tree(X_data, y_encoded)
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        probas = []

        for x in X:
            node = self.root

            while node.left is not None and node.right is not None:
                if x[node.feature_idx] <= node.threshold:
                    node = node.left
                else:
                    node = node.right

            probas.append(node.proba)

        return np.array(probas)

In [12]:
class DecisionTreeRegressor(DecisionTree):
    def __init__(self, max_depth=5, min_samples_split=2):
        super().__init__(max_depth=max_depth, min_samples_split=min_samples_split)

    def _variance(self, y):
        if len(y) == 0:
            return 0.0
        return np.var(y)

    def _find_best_split(self, X, y):
        n_objects, n_features = X.shape

        best_feature = None
        best_threshold = None
        best_gain = -1.0

        parent_var = self._variance(y)

        for feature_idx in range(n_features):
            values = np.unique(X[:, feature_idx])

            if len(values) < 2:
                continue

            thresholds = (values[:-1] + values[1:]) / 2.0

            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                weighted_var = (
                    len(y_left) / len(y) * np.var(y_left) +
                    len(y_right) / len(y) * np.var(y_right)
                )

                gain = parent_var - weighted_var

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold

    def _predict_res(self, y):
        prediction = float(np.mean(y))
        return prediction, None

In [18]:
class ExtraRandomizedTreeClassifier(DecisionTreeClassifier):
    def __init__(self, max_depth=5, min_samples_split=2, random_state=42):
        super().__init__(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            method_split="gini"
        )
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)

    def _find_best_split(self, X, y):
        n_objects, n_features = X.shape

        best_feature = None
        best_threshold = None
        best_gain = -1.0

        parent_gini = self._gini(y)

        classes = np.unique(y)
        if len(classes) < 2:
            return None, None

        for feature_idx in range(n_features):
            x = X[:, feature_idx]

            vmin = np.min(x)
            vmax = np.max(x)

            if vmin == vmax:
                continue

            # случайный порог для данного признака
            threshold = self.rng.uniform(vmin, vmax)

            left_mask = x <= threshold
            right_mask = x > threshold

            y_left = y[left_mask]
            y_right = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gini_left = self._gini(y_left)
            gini_right = self._gini(y_right)

            weighted = (
                len(y_left) / len(y) * gini_left +
                len(y_right) / len(y) * gini_right
            )

            gain = parent_gini - weighted

            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                best_threshold = threshold

        return best_feature, best_threshold

## Task 3-6
3. При использовании модуля DecisionTree необходимо получить коэффициент Джини не менее 0,1 на проверочном наборе данных.
4. Используйте модуль DecisionTreeClassifier из библиотеки sklearn и проверьте его производительность на проверочном наборе данных. Лучше ли он, чем ваш модуль? Если да, то почему?
5. Реализуйте RandomForestClassifier и проверьте его производительность. Вам необходимо улучшить результат одного дерева и получить показатель Джини не менее 0,15 на проверочном наборе данных. Необходимо иметь возможность установить фиксированное начальное значение генератора случайных чисел.
6. Используйте свой класс проектирования DecisionTree для классификатора GBDT. Этот класс должен иметь атрибуты max_depth , number_of_trees и max_features . Вам необходимо вычислить градиент функции потерь бинарной кросс-энтропии и реализовать инкрементальное обучение: обучить следующее дерево, используя результаты предыдущих деревьев.


In [19]:

def gini_score(y_true, y_score):
    auc = roc_auc_score(y_true, y_score)
    return 2 * auc - 1

models = {
    "DecisionTreeClassifier_gini": DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=20,
        method_split="gini"
    ),
    "DecisionTreeClassifier_median": DecisionTreeClassifier(
        max_depth=7,
        min_samples_split=20,
        method_split="median"
    ),
    "DecisionTreeRegressor": DecisionTreeRegressor(
        max_depth=7,
        min_samples_split=20
    ),
    "ExtraRandomizedTreeClassifier": ExtraRandomizedTreeClassifier(
        max_depth=7,
        min_samples_split=20,
        random_state=42
    ),
}


In [20]:

results = []

for name, model in models.items():
    model.fit(X_train.values, y_train.values)

    if hasattr(model, "predict_proba") and not isinstance(model, DecisionTreeRegressor):
        y_score = model.predict_proba(X_valid.values)[:, 1]
    else:
        # для регрессора берём непрерывный output как score
        y_score = model.predict(X_valid.values)

    gini = gini_score(y_valid.values, y_score)
    results.append((name, gini))
    print(f"{name}: Gini = {gini:.6f}")

results_df = pd.DataFrame(results, columns=["model", "gini"]).sort_values("gini", ascending=False)
results_df

DecisionTreeClassifier_gini: Gini = 0.247232
DecisionTreeClassifier_median: Gini = 0.233978
DecisionTreeRegressor: Gini = 0.247232
ExtraRandomizedTreeClassifier: Gini = 0.195708


,model,gini
0,DecisionTreeClassifier_gini,0.247232
2,DecisionTreeRegressor,0.247232
1,DecisionTreeClassifier_median,0.233978
3,ExtraRandomizedTreeClassifier,0.195708


In [21]:
my_valid_gini = results_df.loc[
    results_df["model"] == "DecisionTreeClassifier_gini",
    "gini"
].values[0]

# -----------------------------
# 2. sklearn дерево
# -----------------------------
sk_model = SklearnDecisionTreeClassifier(
    max_depth=7,
    min_samples_split=20,
    criterion="gini",
    random_state=42
)

sk_model.fit(X_train.values, y_train.values)
sk_valid_proba = sk_model.predict_proba(X_valid.values)[:, 1]
sk_valid_gini = gini_score(y_valid.values, sk_valid_proba)


# -----------------------------
# 3. Сравнение
# -----------------------------
print(f"My DecisionTree valid Gini      = {my_valid_gini:.6f}")
print(f"Sklearn DecisionTree valid Gini = {sk_valid_gini:.6f}")

My DecisionTree valid Gini      = 0.247232
Sklearn DecisionTree valid Gini = 0.233808


Моя реализация показала схожее разделение (по значению Gini)

In [22]:
class RandomForestClassifier:

    def __init__(
        self,
        n_estimators=100,
        max_depth=5,
        min_samples_split=2,
        max_features="sqrt",
        random_state=42,
        method_split="gini"
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.method_split = method_split

        self.trees = []
        self.features_idx = []

        self.rng = np.random.default_rng(random_state)


    def _get_n_features(self, n_features):

        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))

        if self.max_features == "log2":
            return max(1, int(np.log2(n_features)))

        if isinstance(self.max_features, int):
            return min(self.max_features, n_features)

        return n_features


    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        n_samples, n_features = X.shape
        n_sub_features = self._get_n_features(n_features)

        self.trees = []
        self.features_idx = []

        for _ in range(self.n_estimators):

            row_idx = self.rng.integers(0, n_samples, n_samples)
            feat_idx = self.rng.choice(
                n_features,
                size=n_sub_features,
                replace=False
            )

            X_boot = X[row_idx][:, feat_idx]
            y_boot = y[row_idx]

            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                method_split=self.method_split
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
            self.features_idx.append(feat_idx)

        return self


    def predict_proba(self, X):
        X = np.asarray(X)
        probas = []
    
        for tree, feat_idx in zip(self.trees, self.features_idx):
            p = tree.predict_proba(X[:, feat_idx])
            probas.append(p)
    
        probas = np.mean(probas, axis=0)
        return probas


    def predict(self, X):

        probas = self.predict_proba(X)

        return np.argmax(probas, axis=1)

In [23]:
rf_model = RandomForestClassifier(
    n_estimators=5,
    max_depth=model.max_depth,
    min_samples_split=model.min_samples_split,
    random_state=42
)

rf_model.fit(X_train.values, y_train.values)

rf_valid_proba = rf_model.predict_proba(X_valid.values)[:, 1]

rf_valid_gini = gini_score(y_valid.values, rf_valid_proba)

print(f"RandomForest valid Gini = {rf_valid_gini:.6f}")

RandomForest valid Gini = 0.263218


In [24]:
class GBDTClassifier:

    def __init__(
        self,
        max_depth=3,
        number_of_trees=100,
        max_features="sqrt",
        learning_rate=0.1,
        min_samples_split=2,
        random_state=42
    ):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.min_samples_split = min_samples_split
        self.random_state = random_state

        self.trees = []
        self.features_idx = []
        self.init_pred = None

        self.rng = np.random.default_rng(random_state)

    def _sigmoid(self, z):
        z = np.clip(z, -50, 50)
        return 1.0 / (1.0 + np.exp(-z))

    def _get_n_features(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        else:
            return n_features

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        n_samples, n_features = X.shape
        n_sub_features = self._get_n_features(n_features)

        self.trees = []
        self.features_idx = []

        # начальный константный logit
        p = np.clip(np.mean(y), 1e-8, 1 - 1e-8)
        self.init_pred = np.log(p / (1 - p))

        # текущие логиты для train
        F = np.full(n_samples, self.init_pred, dtype=float)

        for _ in range(self.number_of_trees):

            # антиградиент BCE по логиту
            residuals = y - self._sigmoid(F)

            # случайное подмножество признаков
            feat_idx = self.rng.choice(
                n_features,
                size=n_sub_features,
                replace=False
            )

            X_sub = X[:, feat_idx]

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split
            )

            tree.fit(X_sub, residuals)

            # incremental update
            update = tree.predict(X_sub)
            F += self.learning_rate * update

            self.trees.append(tree)
            self.features_idx.append(feat_idx)

        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        n_samples = X.shape[0]

        F = np.full(n_samples, self.init_pred, dtype=float)

        for tree, feat_idx in zip(self.trees, self.features_idx):
            F += self.learning_rate * tree.predict(X[:, feat_idx])

        p1 = self._sigmoid(F)
        p0 = 1.0 - p1

        return np.column_stack([p0, p1])

    def predict(self, X):
        proba = self.predict_proba(X)[:, 1]
        return (proba >= 0.5).astype(int)

In [25]:
gbdt = GBDTClassifier(
    max_depth=3,
    number_of_trees=100,
    max_features="sqrt",
    learning_rate=0.1,
    min_samples_split=20,
    random_state=42
)

gbdt.fit(X_train.values, y_train.values)

gbdt_valid_proba = gbdt.predict_proba(X_valid.values)[:, 1]
gbdt_valid_gini = gini_score(y_valid.values, gbdt_valid_proba)

print(f"GBDT valid Gini = {gbdt_valid_gini:.6f}")

GBDT valid Gini = 0.287972


## Task 7
7. Используйте LightGBM, Catboost и XGBoost для обучения на обучающем наборе данных и прогнозирования на валидационном наборе. Изучите документацию библиотек и доработайте алгоритмы для решения поставленной задачи. Отметьте ключевые различия между реализациями. Проанализируйте особенности каждого алгоритма (как работает «категориальный признак» в Catboost, что такое режим DART в XGBoost)? Какая модель GBDT дает наилучший результат? Можете ли вы объяснить, почему?


In [28]:
# -----------------------------
# LightGBM
# -----------------------------
lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=7,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train, y_train)

lgb_valid_proba = lgb_model.predict_proba(X_valid)[:, 1]
lgb_valid_gini = gini_score(y_valid, lgb_valid_proba)


# -----------------------------
# CatBoost
# -----------------------------
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function="Logloss",
    verbose=False,
    random_seed=42
)

[LightGBM] [Info] Number of positive: 2607, number of negative: 20452
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000472 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3872
[LightGBM] [Info] Number of data points in the train set: 23059, number of used features: 29
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.113058 -> initscore=-2.059881
[LightGBM] [Info] Start training from score -2.059881
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [29]:
cat_model.fit(X_train, y_train)

cat_valid_proba = cat_model.predict_proba(X_valid)[:, 1]
cat_valid_gini = gini_score(y_valid, cat_valid_proba)


# -----------------------------
# XGBoost
# -----------------------------
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    eval_metric="auc",
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_valid_proba = xgb_model.predict_proba(X_valid)[:, 1]
xgb_valid_gini = gini_score(y_valid, xgb_valid_proba)


# -----------------------------
# Вывод
# -----------------------------
print(f"LightGBM valid Gini = {lgb_valid_gini:.6f}")
print(f"CatBoost valid Gini = {cat_valid_gini:.6f}")
print(f"XGBoost valid Gini  = {xgb_valid_gini:.6f}")

LightGBM valid Gini = 0.250863
CatBoost valid Gini = 0.259190
XGBoost valid Gini  = 0.258863


## Task 8
Выберите лучшую модель и оцените её производительность на тестовом наборе данных: проверьте значения коэффициента Джини на всех трёх наборах данных для вашей лучшей модели: коэффициент Джини для обучающей выборки, коэффициент Джини для валидационной выборки и коэффициент Джини для тестовой выборки. Замечаете ли вы снижение производительности при сравнении валидационной выборки с тестовой? Переобучается ли ваша модель? Объясните.

In [36]:
models_8 = {
    "DecisionTreeClassifier_gini": models["DecisionTreeClassifier_gini"],
    "DecisionTreeClassifier_median": models["DecisionTreeClassifier_median"],
    "DecisionTreeRegressor": models["DecisionTreeRegressor"],
    "ExtraRandomizedTreeClassifier": models["ExtraRandomizedTreeClassifier"],
    "LightGBM": lgb_model,
    "CatBoost": cat_model,
    "XGBoost": xgb_model,
}

results_8 = []

for name, model_obj in models_8.items():

    if isinstance(model_obj, DecisionTreeRegressor):
        train_score = model_obj.predict(X_train.values)
        valid_score = model_obj.predict(X_valid.values)
        test_score  = model_obj.predict(X_test.values)
    else:
        # для своих деревьев лучше .values, для библиотечных тоже сработает
        train_score = model_obj.predict_proba(X_train.values)[:, 1]
        valid_score = model_obj.predict_proba(X_valid.values)[:, 1]
        test_score  = model_obj.predict_proba(X_test.values)[:, 1]

    train_gini = gini_score(y_train.values, train_score)
    valid_gini = gini_score(y_valid.values, valid_score)
    test_gini  = gini_score(y_test.values, test_score)

    results_8.append({
        "model": name,
        "train_gini": train_gini,
        "valid_gini": valid_gini,
        "test_gini": test_gini
    })

results_8_df = pd.DataFrame(results_8).sort_values("valid_gini", ascending=False)
results_8_df

/home/renat/miniconda3/envs/school21/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/renat/miniconda3/envs/school21/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/renat/miniconda3/envs/school21/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,train_gini,valid_gini,test_gini
5,CatBoost,0.672641,0.259190,0.289427
6,XGBoost,0.867568,0.258863,0.300609
4,LightGBM,0.882954,0.250863,0.293389
0,DecisionTreeClassifier_gini,0.528230,0.247232,0.264220
2,DecisionTreeRegressor,0.528230,0.247232,0.264220
1,DecisionTreeClassifier_median,0.550826,0.233978,0.239781
3,ExtraRandomizedTreeClassifier,0.508321,0.195708,0.233721


модель 1-2 показали стабильные не переобученные результаты, 4-6 немного переобученные но более хорошие

## Task 9*
Реализуйте ExtraTreesClassifier и проверьте его производительность. Необходимо улучшить результат для одного дерева и получить коэффициент Джини не менее 0,12 на проверочном наборе данных.

In [31]:
class ExtraRandomizedTreeClassifier(DecisionTreeClassifier):
    def __init__(self, max_depth=5, min_samples_split=2, random_state=42):
        super().__init__(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            method_split="gini"
        )
        self.random_state = random_state
        self.rng = np.random.default_rng(random_state)

    def _find_best_split(self, X, y):
        n_objects, n_features = X.shape

        best_feature = None
        best_threshold = None
        best_gain = -1.0

        parent_gini = self._gini(y)

        classes = np.unique(y)
        if len(classes) < 2:
            return None, None

        for feature_idx in range(n_features):
            x = X[:, feature_idx]

            vmin = np.min(x)
            vmax = np.max(x)

            if vmin == vmax:
                continue

            # случайный порог для данного признака
            threshold = self.rng.uniform(vmin, vmax)

            left_mask = x <= threshold
            right_mask = x > threshold

            y_left = y[left_mask]
            y_right = y[right_mask]

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gini_left = self._gini(y_left)
            gini_right = self._gini(y_right)

            weighted = (
                len(y_left) / len(y) * gini_left +
                len(y_right) / len(y) * gini_right
            )

            gain = parent_gini - weighted

            if gain > best_gain:
                best_gain = gain
                best_feature = feature_idx
                best_threshold = threshold

        return best_feature, best_threshold

In [32]:
extra_tree = ExtraRandomizedTreeClassifier(
    max_depth=7,
    min_samples_split=20,
    random_state=42
)

extra_tree.fit(X_train.values, y_train.values)

extra_valid_proba = extra_tree.predict_proba(X_valid.values)[:, 1]
extra_valid_gini = gini_score(y_valid.values, extra_valid_proba)

print(f"ExtraRandomizedTree valid Gini = {extra_valid_gini:.6f}")

ExtraRandomizedTree valid Gini = 0.195708
